# Do stress-defense microRNAs change with tree age in centenarian Ginkgo?
**Riddhika D** · ExtremoScan (real-data rebuild) · v1

**Idea:** very old trees survive centuries of drought, cold, UV and pathogens. A simple, testable version of "stress resilience in centenarian plants": do the microRNAs that are **known to control plant stress and defense** shift as *Ginkgo biloba* trees age from 20 to 600 years?

**Data:** NCBI GEO **GSE119213** (Wang et al. 2020, *PNAS*) — small-RNA read counts from the vascular cambium of trees aged **20, 200 and 600 years**, 2 trees per age. Real data, downloaded live. This is the same dataset as the companion Ginkgo-longevity repo, but a **different, pre-registered question** (stress/defense miRNAs, not the juvenility/longevity miRNAs).

### Decided before looking at results
| | |
|---|---|
| **Stress/defense miRNA families (pre-registered)** | miR393 (pathogen/auxin), miR160 & miR167 (auxin/stress), miR398 (oxidative/Cu), miR408 (Cu/drought), miR395 (sulfate/stress), miR399 (phosphate/stress), miR319 & miR159 (drought/ABA), miR402 (abiotic) |
| **Positive control** | **miR397** — already shown (Wang 2020, and our longevity repo) to fall ~77x with age. It must come out strongly age-associated, or the method/data are not trusted |
| **Normalisation** | CPM, log2(CPM+1); keep miRNAs with mean CPM ≥ 10 |
| **Test** | Linear trend of log2 CPM vs log10(age); Benjamini–Hochberg FDR. Set-level: is the stress-miRNA set more age-associated than random miRNAs? |
| **Honest limit** | n = 2 trees/age → exploratory; miRNA family names are cross-species mappings of Ginkgo reads |

Run: **Runtime → Run all** (~1 min). Copy the **SEND THIS TO CLAUDE** box at the end.

In [ ]:
import io, gzip, re, requests
import numpy as np, pandas as pd
from scipy import stats
REPORT = []
def note(s): print(s); REPORT.append(str(s))

URLS = ['https://ftp.ncbi.nlm.nih.gov/geo/series/GSE119nnn/GSE119213/suppl/GSE119213_miRNA.readcount.txt.gz',
        'https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSE119213&format=file&file=GSE119213%5FmiRNA%2Ereadcount%2Etxt%2Egz']
raw = None
for u in URLS:
    try:
        r = requests.get(u, timeout=120); r.raise_for_status()
        raw = pd.read_csv(io.BytesIO(gzip.decompress(r.content)), sep='\t'); break
    except Exception as e: print('  failed', u[:70], str(e)[:80])
assert raw is not None, 'download failed - try again later'
note(f'GSE119213: {raw.shape[0]} miRNA names x {raw.shape[1]-1} samples')

## 1. Samples, ages, normalise

In [ ]:
cols = [c for c in raw.columns if c.endswith('.readcount')]
age = {c: int(re.search(r'VC(\d+)', c).group(1)) for c in cols}
note('samples: ' + ', '.join(f'{c.replace(".readcount","")}={age[c]}y' for c in cols))
C0 = raw.set_index('miRNA_name')[cols].astype(float)
# Merge duplicate names: same miRNA family with near-identical counts in every sample = one molecule
def _fam(n):
    m = re.search(r'(miR\d+)', n, re.I); return m.group(1).lower() if m else n
def _same(a,b): return np.all(np.abs(a-b) <= np.maximum(2, 0.02*np.maximum(a,b)))
order = C0.sum(1).sort_values(ascending=False).index
V = C0.values; pos = {n:i for i,n in enumerate(C0.index)}; TOT = C0.sum(1)
groups, used = [], set()
for n in order:
    if n in used: continue
    g=[n]; used.add(n)
    if TOT[n] >= 20:
        for m in order:
            if m not in used and TOT[m] >= 20 and _fam(m)==_fam(n) and _same(V[pos[n]],V[pos[m]]):
                g.append(m); used.add(m)
    groups.append(g)
C = C0.loc[[g[0] for g in groups]].copy()
ALIAS = {g[0]: g[1:] for g in groups}
note(f'merged {sum(len(g)-1 for g in groups)} duplicate names -> {len(C)} distinct molecules (from {len(C0)})')
CPM = C / C.sum(0) * 1e6
keep = CPM.mean(1) >= 10
L = np.log2(CPM[keep] + 1)
note(f'miRNAs kept (mean CPM >= 10): {int(keep.sum())} of {len(C)}')

## 2. Age trend for every miRNA (then pull out stress + control families)

In [ ]:
def family(n):
    m = re.search(r'(miR\d+)', n, re.I); return m.group(1).lower() if m else None
x = np.log10([age[c] for c in cols])
res = []
for name, y in L.iterrows():
    s = stats.linregress(x, y.values)
    young = y[[c for c in cols if age[c]==min(age.values())]].mean()
    old = y[[c for c in cols if age[c]==max(age.values())]].mean()
    res.append(dict(miRNA=name, family=family(name), mean_CPM=CPM.loc[name].mean(), n_aliases=len(ALIAS.get(name,[])),
                    slope=s.slope, r=s.rvalue, p=s.pvalue, log2FC_600_20=old-young))
R = pd.DataFrame(res)
p = R.p.values; o = np.argsort(p); q = p[o]*len(p)/np.arange(1,len(p)+1)
q = np.minimum.accumulate(q[::-1])[::-1]; R.loc[R.index[o],'FDR'] = np.minimum(q,1)
R = R.sort_values('p'); R.to_csv('ginkgo_stress_mirna_trend.csv', index=False)
note(f'FDR<0.1: {(R.FDR<0.1).sum()} miRNAs')

## 3. Positive control: miR397 must be strongly age-associated

In [ ]:
pc = R[R.family == 'mir397']
note('miR397 (positive control):\n' + (pc[['miRNA','mean_CPM','log2FC_600_20','r','p','FDR']].round(4).to_string(index=False) if len(pc) else 'NOT FOUND'))
ok = len(pc) and (pc.FDR.min() < 0.1)
note('LABEL CHECK: ' + ('PASS - miR397 is age-associated, data/method trusted' if ok
     else 'FAIL - positive control not recovered; do not trust the stress-miRNA result'))

## 4. Do the pre-registered stress/defense miRNAs change with age?

In [ ]:
STRESS = ['mir393','mir160','mir167','mir398','mir408','mir395','mir399','mir319','mir159','mir402']
R['is_stress'] = R.family.isin(STRESS)
found = sorted(set(R[R.is_stress].family))
note(f'stress/defense families present after filtering: {found or "none"}')
St = R[R.is_stress]
if len(St):
    note('stress miRNAs, by p-value:\n' + St[['miRNA','mean_CPM','log2FC_600_20','r','p','FDR','n_aliases']].round(4).to_string(index=False))
    # set-level: are |r| of stress miRNAs larger than random sets of the same size?
    rng = np.random.default_rng(0); obs = St['r'].abs().mean(); n = len(St)
    allr = R['r'].abs().values
    null = np.array([rng.choice(allr, n, replace=False).mean() for _ in range(10000)])
    pset = (1 + (null >= obs).sum()) / 10001
    note(f'SET TEST: mean |r| of {n} stress miRNAs = {obs:.3f} vs random {null.mean():.3f}; empirical p = {pset:.4f}')
    note(f'stress miRNAs passing FDR<0.1 individually: {(St.FDR<0.1).sum()} of {n}')
else:
    note('No pre-registered stress miRNAs survived the CPM filter -> cannot test (reportable null).')

## 5. Figure

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(9,5))
ax.scatter(R.log2FC_600_20, -np.log10(R.p), s=10, c='lightgray', label='all miRNAs')
if R.is_stress.any():
    s = R[R.is_stress]; ax.scatter(s.log2FC_600_20, -np.log10(s.p), s=45, c='crimson', edgecolor='k', label='stress/defense')
    for _,r_ in s.iterrows(): ax.annotate(r_.miRNA, (r_.log2FC_600_20,-np.log10(r_.p)), fontsize=7)
pc = R[R.family=='mir397']
if len(pc): ax.scatter(pc.log2FC_600_20,-np.log10(pc.p), s=60, facecolors='none', edgecolors='navy', label='miR397 (control)')
ax.axvline(0,color='k',lw=.5); ax.set_xlabel('log2 fold change 600y vs 20y'); ax.set_ylabel('-log10 p (age trend)')
ax.legend(); ax.set_title('Stress/defense miRNAs across Ginkgo tree age')
plt.tight_layout(); plt.savefig('ginkgo_stress_mirna.png', dpi=150); plt.show()

## How to read the result
- **Positive control first.** If miR397 is not recovered, ignore everything else.
- A real signal = the stress/defense miRNA set is more age-associated than random miRNAs (set p < 0.05), ideally with specific families passing FDR.
- With only 2 trees per age, **a null is the likely and honest outcome**, and is still worth reporting: it would say stress-defense miRNAs are broadly stable across the Ginkgo lifespan.
- miRNA names are cross-species mappings of Ginkgo reads; treat family calls as approximate.

In [ ]:
print('='*70); print('SEND THIS TO CLAUDE'); print('='*70); print('\n'.join(REPORT)); print('='*70)